## Middleware

- Agent가 행동하기 전이나 후에 중간에서 끼어드는 코드
- before_model : LLM이 호출 전에 실행
- after_model : LLM이 응답한 뒤 실행
- wrap_model_call : LLM 호출 자체를 감싸는 것(동적 모델 선택)
- wrap_tool_call : Tool 실행 자체를 감싸는 것(Tool Error 처리에 사용)


In [1]:
# Before Model

import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import before_model
from langchain.tools import tool
from langchain_nvidia_ai_endpoints import ChatNVIDIA

load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY)


@tool
def calculator(expression: str) -> str:
    """간단한 수식을 계산합니다."""
    try:
        result = eval(expression)
        return str(result)
    except Exception:
        return "계산할 수 없습니다."


@before_model
def log_before_model(state, runtime):
    """LLM 호출 직전에 실행되는 Middleware"""
    print("state : ", state)
    print("LLM 호출 직전")

    messages = state["messages"]

    print("현재 메세지 수 : ", len(messages))

    if messages:
        last_message = messages[-1]
        print("마지막 메세지 : ", last_message)


agent = create_agent(model=llm, tools=[calculator], middleware=[log_before_model])

result = agent.invoke(
    {"messages": [{"role": "user", "content": "25 곱하기 4를 계산해줘."}]}
)

print("최종 결과")

for message in result["messages"]:
    print(type(message).__name__)
    print(message.content)

state :  {'messages': [HumanMessage(content='25 곱하기 4를 계산해줘.', additional_kwargs={}, response_metadata={}, id='b3d4ad06-17ca-46bf-86c6-e622d5490e11')]}
LLM 호출 직전
현재 메세지 수 :  1
마지막 메세지 :  content='25 곱하기 4를 계산해줘.' additional_kwargs={} response_metadata={} id='b3d4ad06-17ca-46bf-86c6-e622d5490e11'
state :  {'messages': [HumanMessage(content='25 곱하기 4를 계산해줘.', additional_kwargs={}, response_metadata={}, id='b3d4ad06-17ca-46bf-86c6-e622d5490e11'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user says "25 곱하기 4를 계산해줘." Means "Please calculate 25 times 4." I should use the calculator function.', 'reasoning': 'The user says "25 곱하기 4를 계산해줘." Means "Please calculate 25 times 4." I should use the calculator function.', '_reasoning_api_fields': ['reasoning_content', 'reasoning'], 'tool_calls': [{'id': 'chatcmpl-tool-b9449b8195eb4ea9', 'type': 'function', 'function': {'name': 'calculator', 'arguments': '{"expression": "25*4"}'}}]}, response_metadata={'role': 'assistant', 'c

In [2]:
# After Model

import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import after_model
from langchain.tools import tool
from langchain_nvidia_ai_endpoints import ChatNVIDIA

load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY)


@tool
def calculator(expression: str) -> str:
    """간단한 수식을 계산합니다."""
    try:
        result = eval(expression)
        return str(result)
    except Exception:
        return "계산할 수 없습니다."


@after_model
def log_after_model(state, runtime):
    """LLM 호출이 끝난 직후 실행되는 Middleware"""
    print("After Model 실행")

    messages = state["messages"]

    print("현재 메세지 수 : ", len(messages))

    if messages:
        last_message = messages[-1]

        print("마지막 메세지 : ", last_message)


agent = create_agent(model=llm, tools=[calculator], middleware=[log_after_model])

result = agent.invoke(
    {"messages": [{"role": "user", "content": "25곱하기 8을 계산해줘"}]}
)

print("최종 결과")
for message in result["messages"]:
    print(type(message).__name__)
    print(message.content)


After Model 실행
현재 메세지 수 :  2
마지막 메세지 :  content='' additional_kwargs={'reasoning_content': 'We need to respond: user says "25곱하기 8을 계산해줘" (Calculate 25 multiplied by 8). They are requesting a calculation. According to tool description, we have a function to do simple arithmetic: functions.calculator. So use the function. We must call functions.calculator with expression "25*8". Then respond with the result.', 'reasoning': 'We need to respond: user says "25곱하기 8을 계산해줘" (Calculate 25 multiplied by 8). They are requesting a calculation. According to tool description, we have a function to do simple arithmetic: functions.calculator. So use the function. We must call functions.calculator with expression "25*8". Then respond with the result.', '_reasoning_api_fields': ['reasoning_content', 'reasoning'], 'tool_calls': [{'id': 'chatcmpl-tool-8686f82694965d8c', 'type': 'function', 'function': {'name': 'calculator', 'arguments': '{"expression": "25*8"}'}}]} response_metadata={'role': 'assistant'

In [5]:
# Wrap Model Call

import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call
from langchain_nvidia_ai_endpoints import ChatNVIDIA

load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY, timeout=600)


@wrap_model_call
def log_model_call(request, handler):
    print("handler : ", handler)
    print("LLM 호출 직전")

    print("현재 메세지 : ", request.state["messages"])

    response = handler(request)

    print("LLM 호출 직후")

    print("LLM 응답:", response)

    return response


agent = create_agent(model=llm, tools=[], middleware=[log_model_call])


result = agent.invoke(
    {"messages": [{"role": "user", "content": "25곱하기 8은 얼마인가요?"}]}
)

for message in result["messages"]:
    print(type(message).__name__)
    print(message.content)

handler :  <function create_agent.<locals>._execute_model_sync at 0x111b3e480>
LLM 호출 직전
현재 메세지 :  [HumanMessage(content='25곱하기 8은 얼마인가요?', additional_kwargs={}, response_metadata={}, id='ab6ea8d5-71d5-43ab-b6db-c8bd73feafc8')]
LLM 호출 직후
LLM 응답: ModelResponse(result=[AIMessage(content='25곱하기 8은 **200**입니다.', additional_kwargs={'reasoning_content': 'The user writes in Korean: "25곱하기 8은 얼마인가요?" Which translates to "What is 25 times 8?" They ask for the product: 25×8 = 200. They want the answer; presumably answer is "200". Should respond in Korean.', 'reasoning': 'The user writes in Korean: "25곱하기 8은 얼마인가요?" Which translates to "What is 25 times 8?" They ask for the product: 25×8 = 200. They want the answer; presumably answer is "200". Should respond in Korean.', '_reasoning_api_fields': ['reasoning_content', 'reasoning']}, response_metadata={'role': 'assistant', 'content': '25곱하기 8은 **200**입니다.', 'refusal': None, 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': []

In [ ]:
# Wrap Model Call Dynamic Model Routing

import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call
from langchain_nvidia_ai_endpoints import ChatNVIDIA

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")

llm_fast = ChatNVIDIA(model=MODEL, api_key=API_KEY)
llm_smart = ChatNVIDIA(model=MODEL, api_key=API_KEY)


@wrap_model_call
def dynamic_model(request, handler):
    print("Wrap Model Call 실행")
    messages = request.state["messages"]

    last_message = messages[-1]

    content = last_message.content

    print("사용자 질문 : ", content)

    print("질문 길이 : ", len(content))

    if len(content) > 20:
        print(" -> Smart 모델 선택")

        request = request.override(model=llm_smart)
    else:
        print(" -> Fast 모델 선택")

        request = request.override(model=llm_fast)

    response = handler(request)

    print("LLM 호출 완료")

    return response


agent = create_agent(model=llm_fast, tools=[], middleware=[dynamic_model])

result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "25곱하기 8은 얼마인지 계산해서 알려주세요"}
        ]
    }
)

for message in result["messages"]:
    print(type(message).__name__)
    print(message.content)

Wrap Model Call 실행
사용자 질문 :  25곱하기 8은 얼마인지 계산해서 알려주세요
질문 길이 :  24
 -> Smart 모델 선택
LLM 호출 완료
HumanMessage
25곱하기 8은 얼마인지 계산해서 알려주세요
AIMessage
25 × 8 = 200입니다.


In [10]:
# Wrap Tool Call

import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.tools import tool
from langchain_nvidia_ai_endpoints import ChatNVIDIA

load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY)


@tool
def calculator(expression: str) -> str:
    """수식을 계산합니다."""

    try:
        result = eval(expression)
        return str(result)
    except Exception:
        return "계산할 수 없습니다."


@wrap_tool_call
def log_tool_call(request, handler):
    print("Tool 호출 직전")
    print("Tool 이름 : ", request.tool_call["name"])
    print("Tool 입력 : ", request.tool_call["args"])

    response = handler(request)

    print("Tool 결과 : ", response)

    return response


agent = create_agent(model=llm, tools=[calculator], middleware=[log_tool_call])

result = agent.invoke(
    {"messages": [{"role": "user", "content": "25곱하기 8을 계산해주세요."}]}
)

for message in result["messages"]:
    print(type(message).__name__)
    print(message.content)


Tool 호출 직전
Tool 이름 :  calculator
Tool 입력 :  {'expression': '25*8'}
Tool 결과 :  content='200' name='calculator' tool_call_id='chatcmpl-tool-af0cc82a76558e65'
HumanMessage
25곱하기 8을 계산해주세요.
AIMessage

ToolMessage
200
AIMessage
25 곱하기 8의 결과는 **200**입니다.


In [13]:
# Wrap Tool Call - Error Handling

import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.tools import tool
from langchain_nvidia_ai_endpoints import ChatNVIDIA

load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY)


@tool
def calculator(expression: str) -> str:
    """수식을 계산합니다."""
    print("Tool 직접 게산 시작")
    return str(eval(expression))


@wrap_tool_call
def handle_tool_error(request, handler):
    print("Tool 호출")
    print("Tool 이름 : ", request.tool_call["name"])
    print("Tool 입력 : ", request.tool_call["args"])

    try:
        response = handler(request)
        print("Tool 정상 실행")
        return response
    except Exception as e:
        print("Tool 실행 중 에러 발생")
        print("Error : ", e)

        return f"Tool 실행에 실패했습니다 : {e}"


agent = create_agent(model=llm, tools=[calculator], middleware=[handle_tool_error])

result = agent.invoke(
    {"messages": [{"role": "user", "content": "10을 0으로 나눠주세요."}]}
)

for message in result["messages"]:
    print(type(message).__name__)
    print(message.content)

Tool 호출
Tool 이름 :  calculator
Tool 입력 :  {'expression': '10/0'}
Tool 직접 게산 시작
Tool 실행 중 에러 발생
Error :  division by zero
HumanMessage
10을 0으로 나눠주세요.
AIMessage

HumanMessage
Tool 실행에 실패했습니다 : division by zero
AIMessage
`10 ÷ 0`는 **정의되지 않은** 연산입니다.  
- 대부분의 수학 시스템에서는 “0으로 나눌 수 없다”는 경고를 내고,  
- 프로그래밍 언어에서 `10/0`을 계산하려 하면 예외가 발생하거나 `Infinity`(무한대)와 같은 특수값을 반환합니다.

따라서 정확히 값을 구할 수 없으며, “정의되지 않음(Undefined)”이라고 표시하는 것이 일반적입니다.


## PIIMiddleware

- Personally Identifiable Information로 개인을 식별할 수 있는 정보
- 이메일, 전화번호, 신용카드 번호, IP주소, 주민등록번호 등등에 대한 처리
- pii_type : 어떤 개인정보를 검사할지
- strategy : 찾은 개인정보를 어떻게 처리할 지


In [31]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_nvidia_ai_endpoints import ChatNVIDIA

load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY)

email_pii = PIIMiddleware("email", strategy="redact", apply_to_input=True)

agent = create_agent(
    model=llm,
    tools=[],
    middleware=[email_pii],
    system_prompt="민감 정보를 안전하게 처리하는 고객지원 보조 에이전트입니다.",
)
mask_config = {}
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "제 이메일은 test@example.com 입니다. 제 이메일을 그대로 말해주세요.",
            }
        ]
    },
)

for message in result["messages"]:
    print(type(message).__name__)
    print(message.content)

HumanMessage
제 이메일은 [REDACTED_EMAIL] 입니다. 제 이메일을 그대로 말해주세요.
AIMessage
I’m sorry, but I can’t provide that.


## Human-in-the-Loop Middleware

- Agent가 중요한 작업을 실행하기 전에 사람에게 승인을 받는 구조


In [40]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import tool
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY)


@tool
def send_email(to: str, content: str) -> str:
    """이메일을 전송합니다."""

    print("받는 사람 : ", to)
    print("내용 : ", content)

    return f"{to}에게 이메일을 성공적으로 보냈습니다."


checkpointer = InMemorySaver()

hitl = HumanInTheLoopMiddleware(interrupt_on={"send_email": True})

agent = create_agent(
    model=llm, tools=[send_email], middleware=[hitl], checkpointer=checkpointer
)

config = {"configurable": {"thread_id": "email_test_001"}}

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "김철수에게 '회의가 오후 3시에 있습니다.' 라고 이메일 보내줘.",
            }
        ]
    },
    config=config,
)

interrupts = result.get("__interrupt__")

print("interrupts")

decision = input("\n이메일을 보내겠습니까? (y = 승인 / n = 거절) : ")

if decision.lower() == "y":
    resume_command = Command(resume={"decisions": [{"type": "approve"}]})
else:
    resume_command = Command(
        resume={
            "decisions": [
                {"type": "reject", "message": "사용자가 이메일 발송을 거절했습니다."}
            ]
        }
    )

result = agent.invoke(resume_command, config=config)


for message in result["messages"]:
    print(type(message).__name__)
    print(message.content)

interrupts
받는 사람 :  김철수
내용 :  회의가 오후 3시에 있습니다.
HumanMessage
김철수에게 '회의가 오후 3시에 있습니다.' 라고 이메일 보내줘.
AIMessage

ToolMessage
김철수에게 이메일을 성공적으로 보냈습니다.
AIMessage
이메일이 김철수에게 전송되었습니다!


## SummarizationMiddleware

- 대화가 너무 길어졌을 때 오래된 대화 내용을 자동으로 요약해서 Context Window를 관리하는 Middleware


In [34]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_nvidia_ai_endpoints import ChatNVIDIA

load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY, timeout=600)

summarization = SummarizationMiddleware(
    model=llm, trigger=("tokens", 4000), keep=("messages", 20)
)

agent = create_agent(model=llm, tools=[], middleware=[summarization])

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "LangChain에서 Middleware가 무엇인지 설명해주세요.",
            }
        ]
    }
)

for message in result["messages"]:
    print(type(message).__name__)
    print(message.content)

HumanMessage
LangChain에서 Middleware가 무엇인지 설명해주세요.
AIMessage
## LangChain에서 Middleware란?  
### ✔️ 핵심 개념  
- **Middleware**는 “중간자” 역할을 하여, Chain/Agent이 실행되는 **전**과 **후**에 특정 로직을 삽입할 수 있는 컴포넌트입니다.  
- Chain, LLM, Agent 등 **핵심 작업(문장 생성, 추론, 메모리 접근)**과 사용자가 원하는 **가공(로깅, 텍스트 변형, 보안, 감시)** 사이를 연결해 주는 “다리” 역할을 합니다.

| 역할 | 설명 | 예시 |
|------|------|------|
| **전 단계 처리** | 입력에 대한 가공, 필터링 | 프롬프트 앞뒤에 토픽 태그 넣기, 사용자 입력 정제 |
| **후 단계 처리** | 출력에 대한 가공, 변환 | 응답에 대한 텍스트 트렝크, JSON 포맷 전환 |
| **가시성** | 실행 로그, 토큰 사용량, Latency 수집 | OpenTelemetry, UI 대시보드 |
| **보안/정책** | 민감 정보 검출, 규제 준수 | 개인정보 마스킹, PII 필터 |

> **핵심**: Middleware는 Chain/Agent 내부 로직을 직접 수정하지 않고도, 재사용 가능한 “작은**한 함수**”들을 **연결**하거나 **교차** 기능을 구현할 수 있게 해 줍니다.

---

## 1. 언어 :  `langchain-core`

LangChain v0.2+에서는 공식적으로 **Middleware API**가 도입되었습니다.  
> **핵심 인터페이스**  
```python
class BaseMiddleware(ABC):
    async def __call__(self, request: Request, next_fn: Callable) -> Response:
        """
        `request` : 실행 전(또는 중간) 상태
        `next_fn` : 다음

In [45]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langgraph.checkpoint.memory import InMemorySaver

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY, timeout=600)

summarization = SummarizationMiddleware(
    model=llm, trigger=("tokens", 300), keep=("messages", 4)
)

checkpointer = InMemorySaver()

agent = create_agent(
    model=llm, tools=[], middleware=[summarization], checkpointer=checkpointer
)

config = {"configurable": {"thread_id": "summary_test_001"}}

questions = [
    "LangChain이 무엇인지 설명해주세요.",
    "LangChain에서 Agent는 무엇인가요?",
    "Middleware는 어떤 역할을 하나요?",
    "RAG는 무엇인가요?",
    "Vector DB는 무엇인가요?",
    "Retriever는 무엇인가요?",
    "Embedding은 무엇인가요?",
    "LangChain과 RAG는 어떻게 연결되나요?",
]

for i, question in enumerate(questions, start=1):
    print("\n")
    print("=" * 60)
    print(f"{i}번째 질문")
    print("=" * 60)

    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question,
                }
            ]
        },
        config=config,
    )

    print("\n현재 메시지 개수:", len(result["messages"]))

    print("\n현재 메시지:")

    for message in result["messages"]:
        print(
            type(message).__name__,
            ":",
            message.content[:200]
            if isinstance(message.content, str)
            else message.content,
        )



1번째 질문

현재 메시지 개수: 2

현재 메시지:
HumanMessage : LangChain이 무엇인지 설명해주세요.
AIMessage : ## LangChain이란?  
**LangChain**은 대형 언어 모델(LLM, Large Language Models)을 활용한 “지능형 애플리케이션”을 손쉽게 만들 수 있도록 설계된 오픈소스 프레임워크입니다.  
Python을 기반으로 하고, OpenAI, Anthropic, Gemini 등 대부분의 LLM 공급자와 호환됩니다.

> **핵심 목표*


2번째 질문

현재 메시지 개수: 4

현재 메시지:
HumanMessage : LangChain이 무엇인지 설명해주세요.
AIMessage : ## LangChain이란?  
**LangChain**은 대형 언어 모델(LLM, Large Language Models)을 활용한 “지능형 애플리케이션”을 손쉽게 만들 수 있도록 설계된 오픈소스 프레임워크입니다.  
Python을 기반으로 하고, OpenAI, Anthropic, Gemini 등 대부분의 LLM 공급자와 호환됩니다.

> **핵심 목표*
HumanMessage : LangChain에서 Agent는 무엇인가요?
AIMessage : ### LangChain Agent — 정의

> **Agent**(에이전트)  
> LLM(대형 언어 모델)이 *“도구(External Tool)”들을 스스로 선택·실행·결과를 받아 다음 행동을 결정해 나가는* 실행 엔진.

즉, Agent는 “LLM + Tool + Decision Cycle”의 프로세스를 추상화한 객체입니다.  
단순히 한 번에 텍스트


3번째 질문

현재 메시지 개수: 6

현재 메시지:
HumanMessage : Here is a summary of the conversation to date:

## SESSION INTENT
The user requests an explanation of LangChain.

## SUMMARY
The us

## ModelCallLimitMiddleware

- LLM 호출 횟수를 제한


In [50]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain.tools import tool

load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY, timeout=600)


@tool
def calculator(expression: str) -> str:
    """수식을 계산합니다."""
    print("입력 : ", expression)

    result = eval(expression)

    print("결과 : ", result)
    return str(result)


model_limit = ModelCallLimitMiddleware(thread_limit=2)

agent = create_agent(model=llm, tools=[calculator], middleware=[model_limit])

result = agent.invoke(
    {"messages": [{"role": "user", "content": "25 곱하기 8을 계산해주세요."}]}
)

for message in result["messages"]:
    print(type(message).__name__)
    print(message.content)

입력 :  25 * 8
결과 :  200
HumanMessage
25 곱하기 8을 계산해주세요.
AIMessage

ToolMessage
200
AIMessage
25 곱하기 8의 결과는 **200**입니다.


## ToolCallLimitMiddleware

- Tool 호출 횟수를 제한


In [55]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langchain.tools import tool
from langchain_nvidia_ai_endpoints import ChatNVIDIA

load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY, timeout=600)


@tool
def calculator(expression: str) -> str:
    """수식을 계산합니다."""

    print("입력 : ", expression)

    result = eval(expression)

    print("결과 : ", result)

    return str(result)


tool_limit = ToolCallLimitMiddleware(tool_name="calculator", thread_limit=1)

agent = create_agent(model=llm, tools=[calculator], middleware=[tool_limit])

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "다음 계산을 순서대로 해주세요. 먼저 툴을 사용해서 25 곱하기 8을 계산하고, 그 결과에 툴을 사용해서 10을 더해서 최종 결과를 알려주세요.",
            }
        ]
    }
)

for message in result["messages"]:
    print("\n", type(message).__name__)
    print(message.content)

입력 :  25*8
결과 :  200

 HumanMessage
다음 계산을 순서대로 해주세요. 먼저 툴을 사용해서 25 곱하기 8을 계산하고, 그 결과에 툴을 사용해서 10을 더해서 최종 결과를 알려주세요.

 AIMessage


 ToolMessage
200

 AIMessage


 ToolMessage
Tool call limit exceeded. Do not call 'calculator' again.

 AIMessage
최종 결과는 **210**입니다.


## ModelFallbackMiddleware

- 주 모델이 실패하면 다른 모델로 자동 전환하는 Middleware


In [58]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import ModelFallbackMiddleware
from langchain_nvidia_ai_endpoints import ChatNVIDIA

load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")

primary_model = ChatNVIDIA(model="존재하지 않는 모델", api_key=API_KEY, timeout=600)

fallback_model = ChatNVIDIA(model=MODEL, api_key=API_KEY, timeout=600)

fallback = ModelFallbackMiddleware(fallback_model)

agent = create_agent(model=primary_model, tools=[], middleware=[fallback])

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "LangChain이 무엇인지 한 문장으로 설명해주세요.",
            }
        ]
    }
)

for message in result["messages"]:
    print(type(message).__name__)
    print(message.content)

/Users/parkchanryong/Desktop/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:257: UserWarning: Model 존재하지 않는 모델 is unknown, check `available_models`. Inference may fail.
  warnings.warn(


HumanMessage
LangChain이 무엇인지 한 문장으로 설명해주세요.
AIMessage
LangChain은 외부 도구와 기억 기능을 결합해 언어 모델 기반 애플리케이션을 손쉽게 구축할 수 있는 오픈소스 프레임워크입니다.
